In [4]:
# Install packages jika belum tersedia
if (!require("nnet")) install.packages("nnet", repos = "https://cloud.r-project.org/")
if (!require("dplyr")) install.packages("dplyr", repos = "https://cloud.r-project.org/")
if (!require("tidyr")) install.packages("tidyr", repos = "https://cloud.r-project.org/")
if (!require("broom")) install.packages("broom", repos = "https://cloud.r-project.org/")
if (!require("car")) install.packages("car", repos = "https://cloud.r-project.org/")
if (!require("DescTools")) install.packages("DescTools", repos = "https://cloud.r-project.org/")

library(nnet)
library(dplyr)
library(tidyr)
library(broom)
library(car)
library(DescTools)

Loading required package: DescTools

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
"there is no package called 'DescTools'"
Installing package into 'C:/Users/LENOVO/AppData/Local/R/win-library/4.5'
(as 'lib' is unspecified)



also installing the dependencies 'rootSolve', 'lmom', 'expm', 'Exact', 'gld'




package 'rootSolve' successfully unpacked and MD5 sums checked
package 'lmom' successfully unpacked and MD5 sums checked
package 'expm' successfully unpacked and MD5 sums checked
package 'Exact' successfully unpacked and MD5 sums checked
package 'gld' successfully unpacked and MD5 sums checked
package 'DescTools' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\LENOVO\AppData\Local\Temp\Rtmp67J1Ot\downloaded_packages


Warning message:
"package 'DescTools' was built under R version 4.5.3"

Attaching package: 'DescTools'


The following object is masked from 'package:car':

    Recode




## 1. Persiapan Data
Load data, konversi ke long format, dan tentukan level referensi.

In [5]:
# Load data agregat
df_agg <- read.csv("alligator_food_choice.csv")
head(df_agg)

# Konversi ke long format
df_long <- df_agg %>%
  pivot_longer(
    cols = c(Fish, Invertebrate, Reptile, Bird, Other),
    names_to = "Food_Choice",
    values_to = "count"
  ) %>%
  uncount(weights = count)

# Tentukan level referensi
df_long$Food_Choice <- relevel(factor(df_long$Food_Choice), ref = "Fish")
df_long$Lake <- relevel(factor(df_long$Lake), ref = "George")
df_long$Gender <- relevel(factor(df_long$Gender), ref = "Male")
df_long$Size_m <- relevel(factor(df_long$Size_m), ref = "<=2.3")

# Cek struktur data
str(df_long)
table(df_long$Food_Choice)

,Lake,Gender,Size_m,Fish,Invertebrate,Reptile,Bird,Other
,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>
1,Hancock,Male,<=2.3,7,1,0,0,5
2,Hancock,Male,>2.3,4,0,0,1,2
3,Hancock,Female,<=2.3,16,3,2,2,3
4,Hancock,Female,>2.3,3,0,1,2,3
5,Oklawaha,Male,<=2.3,2,2,0,0,1
6,Oklawaha,Male,>2.3,13,7,6,0,0


tibble [219 × 4] (S3: tbl_df/tbl/data.frame)
 $ Lake       : Factor w/ 4 levels "George","Hancock",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ Gender     : Factor w/ 2 levels "Male","Female": 1 1 1 1 1 1 1 1 1 1 ...
 $ Size_m     : Factor w/ 2 levels "<=2.3",">2.3": 1 1 1 1 1 1 1 1 1 1 ...
 $ Food_Choice: Factor w/ 5 levels "Fish","Bird",..: 1 1 1 1 1 1 1 3 4 4 ...



        Fish         Bird Invertebrate        Other      Reptile 
          94           13           61           32           19 

## 2. Fit Model Regresi Logistik Multinomial

In [15]:
# Fit model multinomial
model <- multinom(Food_Choice ~ Lake + Gender + Size_m, data = df_long, trace = FALSE)
summary(model)

Call:
multinom(formula = Food_Choice ~ Lake + Gender + Size_m, data = df_long, 
    trace = FALSE)

Coefficients:
             (Intercept) LakeHancock LakeOklawaha LakeTrafford GenderFemale
Bird          -3.0386187   0.5753859  -0.55029893     1.237111    0.6064079
Invertebrate  -0.2939452  -1.7804428   0.91320520     1.155850    0.4629561
Other         -1.6833164   0.7665839   0.02605831     1.557776    0.2525889
Reptile       -4.0435542   1.1294399   2.53020293     3.061016    0.6275746
             Size_m>2.3
Bird          0.7302265
Invertebrate -1.3362685
Other        -0.2905753
Reptile       0.5570452

Std. Errors:
             (Intercept) LakeHancock LakeOklawaha LakeTrafford GenderFemale
Bird           0.8319419   0.7952339    1.2098974    0.8661140    0.6888548
Invertebrate   0.3552706   0.6232018    0.4761174    0.4927870    0.3955221
Other          0.5209746   0.5685509    0.7777721    0.6256744    0.4663471
Reptile        1.1839215   1.1927674    1.1220866    1.1296991    0.

## 3. Parameter Estimates
Tabel koefisien (log-odds), standard error, z-value, p-value (z-test), **Wald Chi-square**, **p-Wald**, Odds Ratio, dan 95% Confidence Interval.

In [12]:
s <- summary(model)
coefs <- s$coefficients
se <- s$standard.errors
z <- coefs / se
wald_chisq <- z^2
p_wald <- 1 - pchisq(wald_chisq, df = 1)
or <- exp(coefs)
ci <- exp(confint(model))

outcome_levels <- rownames(coefs)
var_names <- colnames(coefs)

for (lvl in outcome_levels) {
  cat("\n============================================================\n")
  cat("  Parameter Estimates:", lvl, "vs Fish (Reference)\n")
  cat("============================================================\n")
  
  sig_fmt <- function(p) {
    p_rounded <- round(p, 3)
    ifelse(p < 0.001, "<.001", sub("^0\\.", ".", format(p_rounded, nsmall = 3)))
  }
  
  res <- data.frame(
    Parameter = var_names,
    B = format(round(coefs[lvl, ], 3), nsmall = 3),
    `Std.Error` = format(round(se[lvl, ], 3), nsmall = 3),
    Wald = format(round(wald_chisq[lvl, ], 3), nsmall = 3),
    df = 1,
    Sig. = sig_fmt(p_wald[lvl, ]),
    `Exp(B)` = format(round(or[lvl, ], 3), nsmall = 3),
    `95% CI Lower` = format(round(ci[, "2.5 %", lvl], 3), nsmall = 3),
    `95% CI Upper` = format(round(ci[, "97.5 %", lvl], 3), nsmall = 3),
    check.names = FALSE,
    stringsAsFactors = FALSE
  )
  
  print(res, row.names = FALSE)
}


  Parameter Estimates: Bird vs Fish (Reference)
    Parameter      B Std.Error   Wald df  Sig. Exp(B) 95% CI Lower 95% CI Upper
  (Intercept) -3.039     0.832 13.340  1 <.001  0.048        0.009        0.245
  LakeHancock  0.575     0.795  0.524  1  .469  1.778        0.374        8.449
 LakeOklawaha -0.550     1.210  0.207  1  .649  0.577        0.054        6.178
 LakeTrafford  1.237     0.866  2.040  1  .153  3.446        0.631       18.815
 GenderFemale  0.606     0.689  0.775  1  .379  1.834        0.475        7.075
   Size_m>2.3  0.730     0.652  1.253  1  .263  2.076        0.578        7.454

  Parameter Estimates: Invertebrate vs Fish (Reference)
    Parameter      B Std.Error   Wald df Sig. Exp(B) 95% CI Lower 95% CI Upper
  (Intercept) -0.294     0.355  0.685  1 .408  0.745        0.371        1.495
  LakeHancock -1.780     0.623  8.162  1 .004  0.169        0.050        0.572
 LakeOklawaha  0.913     0.476  3.679  1 .055  2.492        0.980        6.337
 LakeTrafford  1.1

## 4. Likelihood Ratio Test (LRT) per Variabel

In [13]:
# LRT untuk setiap variabel menggunakan car::Anova
lrt <- Anova(model, test = "LR")

cat("============================================================\n")
cat("  Likelihood Ratio Tests (LRT) per Variabel\n")
cat("============================================================\n")

sig_fmt <- function(p) {
  p_rounded <- round(p, 3)
  ifelse(p < 0.001, "<.001", sub("^0\\.", ".", format(p_rounded, nsmall = 3)))
}

lrt_df <- data.frame(
  Effect = rownames(lrt),
  `LR Chisq` = format(round(lrt$`LR Chisq`, 3), nsmall = 3),
  df = lrt$Df,
  `Sig.` = sig_fmt(lrt$`Pr(>Chisq)`),
  check.names = FALSE,
  stringsAsFactors = FALSE
)

print(lrt_df, row.names = FALSE)

  Likelihood Ratio Tests (LRT) per Variabel
 Effect LR Chisq df  Sig.
   Lake   50.318 12 <.001
 Gender    2.215  4  .696
 Size_m   17.600  4  .001


## 5. Goodness of Fit
Deviance, AIC, Pearson Chi-square, Pseudo R-squared, dan Classification Accuracy.

In [9]:
# Deviance dan AIC
cat("Deviance:", deviance(model), "\n")
cat("AIC:", AIC(model), "\n")

# Pearson Chi-square Goodness of Fit
# Agregasi observed counts per kombinasi
observed <- df_agg %>%
  pivot_longer(
    cols = c(Fish, Invertebrate, Reptile, Bird, Other),
    names_to = "Food_Choice",
    values_to = "count"
  )

# Kombinasi unik
newdata <- df_agg %>% select(Lake, Gender, Size_m) %>% distinct()
pred_probs <- predict(model, newdata = newdata, type = "probs")
pred_df <- as.data.frame(pred_probs)
pred_df <- cbind(newdata, pred_df)

# Total N per kombinasi
N_comb <- df_long %>% count(Lake, Gender, Size_m) %>% rename(N = n)

# Expected counts
expected <- pred_df %>%
  pivot_longer(
    cols = c(Fish, Invertebrate, Reptile, Bird, Other),
    names_to = "Food_Choice",
    values_to = "prob"
  ) %>%
  left_join(N_comb, by = c("Lake", "Gender", "Size_m")) %>%
  mutate(expected = prob * N)

# Merge
gof_df <- observed %>%
  left_join(expected, by = c("Lake", "Gender", "Size_m", "Food_Choice"))

# Pearson statistic
pearson_chisq <- sum((gof_df$count - gof_df$expected)^2 / gof_df$expected)
df_gof <- nrow(gof_df) - length(coef(model)) - 1
p_pearson <- 1 - pchisq(pearson_chisq, df = max(df_gof, 1))

cat("Pearson Chi-square:", pearson_chisq, "\n")
cat("df:", df_gof, "\n")
cat("p-value:", p_pearson, "\n")

# Pseudo R-squared
PseudoR2(model, c("McFadden", "CoxSnell", "Nagelkerke"))

# Classification accuracy
pred_class <- predict(model)
conf_matrix <- table(Predicted = pred_class, Actual = df_long$Food_Choice)
accuracy <- sum(diag(conf_matrix)) / sum(conf_matrix)

cat("\nConfusion Matrix:\n")
print(conf_matrix)
cat("\nClassification Accuracy:", accuracy, "\n")

Deviance: 537.8655 
AIC: 585.8655 
Pearson Chi-square: 52.56849 
df: 55 
p-value: 0.5680797 


Warning message in PseudoR2(model, c("McFadden", "CoxSnell", "Nagelkerke")):
"Could not find model or data element of multinom object for evaluating PseudoR2 null model. Will fit null model with new evaluation of 'df_long'. Ensure object has not changed since initial call, or try running multinom with 'model = TRUE'"


McFadden   CoxSnell Nagelkerke 
 0.1100290  0.2618744  0.2795755


Confusion Matrix:
              Actual
Predicted      Fish Bird Invertebrate Other Reptile
  Fish           81   12           29    23      15
  Bird            0    0            0     0       0
  Invertebrate   13    1           31     9       4
  Other           0    0            0     0       0
  Reptile         0    0            1     0       0

Classification Accuracy: 0.5114155 


## 6. Predicted Probabilities

In [10]:
# Tabel predicted probabilities
pred_table <- pred_df %>%
  left_join(N_comb, by = c("Lake", "Gender", "Size_m")) %>%
  arrange(Lake, Gender, Size_m)

# Round for display
pred_table[, c("Fish", "Invertebrate", "Reptile", "Bird", "Other")] <- 
  round(pred_table[, c("Fish", "Invertebrate", "Reptile", "Bird", "Other")], 4)

print(pred_table)

       Lake Gender Size_m   Fish   Bird Invertebrate  Other Reptile  N
1    George Female  <=2.3 0.3931 0.0345       0.4655 0.0940  0.0129 14
2    George Female   >2.3 0.5781 0.1054       0.1799 0.1034  0.0331 10
3    George   Male  <=2.3 0.5009 0.0240       0.3733 0.0930  0.0088 27
4    George   Male   >2.3 0.6827 0.0679       0.1337 0.0948  0.0209 12
5   Hancock Female  <=2.3 0.5071 0.0792       0.1012 0.2610  0.0515 26
6   Hancock Female   >2.3 0.5158 0.1672       0.0271 0.1985  0.0915  9
7   Hancock   Male  <=2.3 0.6006 0.0512       0.0755 0.2402  0.0326 13
8   Hancock   Male   >2.3 0.6236 0.1102       0.0206 0.1865  0.0591  7
9  Oklawaha Female  <=2.3 0.2146 0.0109       0.6333 0.0527  0.0885 15
10 Oklawaha Female   >2.3 0.3592 0.0378       0.2786 0.0659  0.2585  2
11 Oklawaha   Male  <=2.3 0.3034 0.0084       0.5636 0.0578  0.0668  5
12 Oklawaha   Male   >2.3 0.4825 0.0277       0.2356 0.0688  0.1854 26
13 Trafford Female  <=2.3 0.1449 0.0439       0.5451 0.1645  0.1016 12
14 Tra

## 7. Interpretasi

### Signifikansi Variabel (LRT)
Berdasarkan Likelihood Ratio Test (LRT) dengan `car::Anova()`:

- **Lake**: **Signifikan** (LR Chisq = 50.32, df = 12, p < 0.001). Terdapat perbedaan sangat signifikan dalam pilihan makanan buaya antar danau. Danau mempengaruhi ketersediaan jenis mangsa sehingga buaya di habitat yang berbeda cenderung memilih makanan yang berbeda.
- **Gender**: **Tidak signifikan** (LR Chisq = 2.21, df = 4, p = 0.696). Gender tidak memberikan pengaruh signifikan terhadap pilihan makanan buaya dalam data ini.
- **Size_m**: **Signifikan** (LR Chisq = 17.60, df = 4, p = 0.001). Ukuran tubuh buaya secara signifikan mempengaruhi pilihan makanan.

### Wald Test
Wald test (z-test dan Wald Chi-square) menunjukkan signifikansi masing-masing koefisien. Hasilnya konsisten dengan LRT: prediktor signifikan pada level individu memiliki p-Wald < 0.05.

### Odds Ratio (OR) Signifikan
Odds Ratio menunjukkan perubahan odds memilih kategori makanan tertentu (vs Fish) ketika prediktor berubah dari level referensi.

**Invertebrate vs Fish**
- Buaya di danau Hancock memiliki odds memilih Invertebrate **0.17 kali** (OR = 0.169, p = 0.004) dibandingkan buaya di danau George. Ini berarti odds memilih Invertebrate jauh lebih rendah di Hancock.
- Buaya di danau Trafford memiliki odds memilih Invertebrate **3.18 kali** (OR = 3.177, p = 0.019) dibandingkan buaya di danau George.
- Buaya berukuran >2.3 meter memiliki odds memilih Invertebrate **0.26 kali** (OR = 0.263, p = 0.001) dibandingkan buaya berukuran <=2.3 meter. Buaya besar jarang memilih invertebrata.

**Other vs Fish**
- Buaya di danau Trafford memiliki odds memilih Other **4.75 kali** (OR = 4.748, p = 0.013) dibandingkan buaya di danau George.

**Reptile vs Fish**
- Buaya di danau Oklawaha memiliki odds memilih Reptile **12.56 kali** (OR = 12.56, p = 0.024) dibandingkan buaya di danau George.
- Buaya di danau Trafford memiliki odds memilih Reptile **21.35 kali** (OR = 21.35, p = 0.007) dibandingkan buaya di danau George.

**Bird vs Fish**
- Tidak ada prediktor yang signifikan secara individual, namun intercept menunjukkan odds dasar Bird sangat rendah (OR = 0.048, p < 0.001) dibandingkan Fish.

### Goodness of Fit
- Pearson Chi-square = 52.57 (df = 55, p = 0.568) menunjukkan model memiliki **goodness of fit yang memadai** (tidak ada penyimpangan signifikan antara observed dan expected).
- Pseudo R²: McFadden = 0.110, Cox & Snell = 0.262, Nagelkerke = 0.280. Ini menunjukkan model menjelaskan variansi pilihan makanan secara moderat.
- Classification Accuracy = 51.1%. Meskipun tidak tinggi, ini lebih baik dari baseline (klassifikasi semua sebagai Fish = 42.9%).

### Pola Pilihan Makanan (Predicted Probabilities)
Berdasarkan tabel predicted probabilities:
- **Fish** tetap menjadi pilihan dominan untuk sebagian besar kombinasi, terutama di danau George (probabilitas 0.39–0.68) dan Hancock (0.51–0.62).
- **Invertebrate** memiliki probabilitas tertinggi di danau Oklawaha dan Trafford untuk buaya kecil (<=2.3), terutama pada betina Oklawaha (probabilitas 0.63).
- **Reptile** menunjukkan probabilitas yang lebih tinggi pada buaya besar (>2.3 m) di danau Oklawaha (0.19–0.26) dan Trafford (0.20–0.26), mengindikasikan peralihan diet ke mangsa lebih besar seiring pertumbuhan.
- **Bird** dan **Other** memiliki probabilitas relatif rendah di semua kombinasi.

### Kesimpulan
Model regresi logistik multinomial menunjukkan bahwa **danau (Lake)** dan **ukuran tubuh (Size_m)** adalah faktor signifikan yang memengaruhi pilihan makanan buaya, sementara **gender** tidak berpengaruh signifikan. Buaya di danau Oklawaha dan Trafford cenderung lebih sering memilih Invertebrate dan Reptile dibandingkan Fish, sedangkan buaya di George dan Hancock lebih dominan memilih Fish. Buaya berukuran besar (>2.3 m) cenderung mengurangi konsumsi invertebrata dan meningkatkan kemungkinan memilih reptil, mencerminkan perubahan diet seiring pertumbuhan.